In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 1) SET PATHS
# =========================================================
raw_path = Path("../data/raw/YRBS_2007.csv")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = processed_dir / "yrbs_cleaned.csv"
recoded_path = processed_dir / "yrbs_recoded.csv"

# =========================================================
# 2) LOAD DATA
# =========================================================
df = pd.read_csv(raw_path)

print("=== ORIGINAL DATA INFO ===")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

# Keep a copy of original
df_clean = df.copy()

# =========================================================
# 3) APPROVED VARIABLES FOR THIS PROJECT
# =========================================================
approved_behavior = [
    "EverCigaretteUse",
    "SadOrHopeless",
    "CurrentCigaretteUse",
    "CurrentAlcoholUse"
]

approved_continuous = [
    "HowTallAreYouWithoutShoesInMeters",
    "HowMuchDoYouWeighWithoutShoesInKG",
    "BMIPCT"
]

approved_vars = approved_behavior + approved_continuous

# Check which approved variables exist
print("\n=== VARIABLE CHECK ===")
for col in approved_vars:
    if col in df_clean.columns:
        print(f"[OK] {col}")
    else:
        print(f"[MISSING] {col}")

# =========================================================
# 4) BASIC DATA CHECK
# =========================================================
print("\n=== MISSING VALUES COUNT ===")
missing_table = df_clean[approved_vars].isna().sum()
print(missing_table)

print("\n=== DATA TYPES ===")
print(df_clean[approved_vars].dtypes)

# =========================================================
# 5) CONVERT RELEVANT VARIABLES TO NUMERIC
# =========================================================
for col in approved_vars:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("\n=== AFTER NUMERIC CONVERSION ===")
print(df_clean[approved_vars].dtypes)

# =========================================================
# 6) VALUE CHECK FOR BEHAVIOR VARIABLES
#    (these are usually coded as integers)
# =========================================================
print("\n=== ORIGINAL CODE FREQUENCIES (Behavior Variables) ===")
for col in approved_behavior:
    if col in df_clean.columns:
        print(f"\n--- {col} ---")
        print(df_clean[col].value_counts(dropna=False).sort_index())

# =========================================================
# 7) PLAUSIBILITY CHECK FOR CONTINUOUS VARIABLES
# =========================================================
print("\n=== CONTINUOUS VARIABLE RANGE CHECK ===")

# Create a copy for cleaning continuous values
for col in approved_continuous:
    if col in df_clean.columns:
        print(f"\n--- {col} ---")
        print("Min:", df_clean[col].min())
        print("Max:", df_clean[col].max())

# Optional plausibility rules
# Adjust only if your instructor gave different valid ranges
if "HowTallAreYouWithoutShoesInMeters" in df_clean.columns:
    df_clean.loc[
        ~df_clean["HowTallAreYouWithoutShoesInMeters"].between(1.0, 2.5, inclusive="both"),
        "HowTallAreYouWithoutShoesInMeters"
    ] = np.nan

if "HowMuchDoYouWeighWithoutShoesInKG" in df_clean.columns:
    df_clean.loc[
        ~df_clean["HowMuchDoYouWeighWithoutShoesInKG"].between(20, 250, inclusive="both"),
        "HowMuchDoYouWeighWithoutShoesInKG"
    ] = np.nan

if "BMIPCT" in df_clean.columns:
    df_clean.loc[
        ~df_clean["BMIPCT"].between(0, 100, inclusive="both"),
        "BMIPCT"
    ] = np.nan

print("\n=== CONTINUOUS VARIABLE RANGE CHECK AFTER CLEANING ===")
for col in approved_continuous:
    if col in df_clean.columns:
        print(f"\n--- {col} ---")
        print("Min:", df_clean[col].min())
        print("Max:", df_clean[col].max())
        print("Missing:", df_clean[col].isna().sum())

# =========================================================
# 8) SAVE CLEANED DATA
# =========================================================
df_clean.to_csv(cleaned_path, index=False)
print(f"\n[Saved] Cleaned data -> {cleaned_path}")

# =========================================================
# 9) CREATE RECODED DATASET
# =========================================================
df_recoded = df_clean.copy()

# -----------------------------
# Behavior Variable Recoding
# -----------------------------
# EverCigaretteUse: success = 1, failure = 2
if "EverCigaretteUse" in df_recoded.columns:
    df_recoded["EverCigaretteUse_binary"] = np.where(
        df_recoded["EverCigaretteUse"] == 1, 1,
        np.where(df_recoded["EverCigaretteUse"] == 2, 0, np.nan)
    )

# SadOrHopeless: success = 1, failure = 2
if "SadOrHopeless" in df_recoded.columns:
    df_recoded["SadOrHopeless_binary"] = np.where(
        df_recoded["SadOrHopeless"] == 1, 1,
        np.where(df_recoded["SadOrHopeless"] == 2, 0, np.nan)
    )

# CurrentCigaretteUse: success = 2–7, failure = 1
if "CurrentCigaretteUse" in df_recoded.columns:
    df_recoded["CurrentCigaretteUse_binary"] = np.where(
        df_recoded["CurrentCigaretteUse"].between(2, 7, inclusive="both"), 1,
        np.where(df_recoded["CurrentCigaretteUse"] == 1, 0, np.nan)
    )

# CurrentAlcoholUse: success = 2–7, failure = 1
if "CurrentAlcoholUse" in df_recoded.columns:
    df_recoded["CurrentAlcoholUse_binary"] = np.where(
        df_recoded["CurrentAlcoholUse"].between(2, 7, inclusive="both"), 1,
        np.where(df_recoded["CurrentAlcoholUse"] == 1, 0, np.nan)
    )

# -----------------------------
# Optional: Add benchmark columns
# -----------------------------
# These are not required, but can help later
df_recoded["benchmark_EverCigaretteUse"] = 0.50
df_recoded["benchmark_SadOrHopeless"] = 0.30
df_recoded["benchmark_CurrentCigaretteUse"] = 0.20
df_recoded["benchmark_CurrentAlcoholUse"] = 0.35
df_recoded["benchmark_Height"] = 1.70
df_recoded["benchmark_Weight"] = 68.0
df_recoded["benchmark_BMIPCT"] = 65.0

# =========================================================
# 10) CHECK RECODED VARIABLES
# =========================================================
print("\n=== RECODED VARIABLE CHECK ===")

recoded_vars = [
    "EverCigaretteUse_binary",
    "SadOrHopeless_binary",
    "CurrentCigaretteUse_binary",
    "CurrentAlcoholUse_binary"
]

for col in recoded_vars:
    if col in df_recoded.columns:
        print(f"\n--- {col} ---")
        print(df_recoded[col].value_counts(dropna=False).sort_index())

# =========================================================
# 11) SAVE RECODED DATA
# =========================================================
df_recoded.to_csv(recoded_path, index=False)
print(f"\n[Saved] Recoded data -> {recoded_path}")

# =========================================================
# 12) FINAL SAMPLE SIZE CHECK FOR EACH PROJECT VARIABLE
# =========================================================
print("\n=== FINAL USABLE SAMPLE SIZE ===")

for col in approved_vars:
    if col in df_recoded.columns:
        usable_n = df_recoded[col].notna().sum()
        print(f"{col}: n = {usable_n}")

for col in recoded_vars:
    if col in df_recoded.columns:
        usable_n = df_recoded[col].notna().sum()
        print(f"{col}: n = {usable_n}")

=== ORIGINAL DATA INFO ===
Shape: (14041, 103)

Columns:
['RaceEth', 'HowOldAreYou', 'WhatIsYourSex', 'InWhatGradeAreYou', 'AreYouHispanicOrLatino', 'WhatIsYourRace', 'HowTallAreYouWithoutShoesInMeters', 'HowMuchDoYouWeighWithoutShoesInKG', 'BicyleHelmetUse', 'SeatBeltUse', 'RidingWithADrinkingDriver', 'DrinkingAndDriving', 'WeaponCarrying', 'GunCarryingPast12Mos', 'WeaponCarryingAtSchool', 'SafetyConcernsAtSchool', 'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty', 'StolenOrDamagedYourProperty', 'PhysicalFighting', 'InjuredIFight', 'PhysicalFightingAtSchool', 'BoyfriendGirlfriendPhysicallyHurt', 'ForcedSexualIntercourse', 'SadOrHopeless', 'ConsideredSuicide', 'MadeASuicidePlan', 'AttemptedSuicide', 'InjuriousSuicide', 'EverCigaretteUse', 'InitiationSmokingWholeCigarette', 'CurrentCigaretteUse', 'SmokedMoreThan10Cigarettes', 'HowObtainedCigarettes', 'SmokeOnSchoolProperty', 'EverSmokedDailyFor30Days', 'EverSmokingCessation', 'CurrentSmokelessTobaccoUse', 'CurrentSmokelessTobaccoOnSc